# Regression-testing LLM-as-judge evals in CI with OpenAI

This notebook demonstrates a simple, CI-friendly pattern using OpenAI's chat completions:

1. Use `response_format: json_object` to score outputs against a rubric.
2. Capture scores in a snapshot file committed to your repo.
3. Fail CI when scores drop below tolerance.

The pattern works with any pytest-shaped runner. We use [evalcheck](https://github.com/Boiga7/evalcheck) here for the snapshot diff — you could roll your own with `pytest` + a JSON file.

In [ ]:
%pip install --quiet openai evalcheck pytest

## A thing to evaluate

Any LLM-driven function works. Summarising an article, writing a SQL query, classifying a ticket. Here: a one-sentence summariser.

In [ ]:
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY env var

def summarise(article: str) -> str:
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "Summarise the article in one sentence."},
            {"role": "user", "content": article},
        ],
    )
    return completion.choices[0].message.content

## A homegrown OpenAI-as-judge metric

If you don't want a third-party eval library, this 15-line function is the whole thing. `response_format: {"type": "json_object"}` forces JSON output so you don't have to fight the model into a parseable shape.

evalcheck ships an equivalent metric (`faithfulness`) — this is just to show what's underneath the abstraction.

In [ ]:
import json

def faithfulness_score(output: str, context: str) -> float:
    judgment = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": (
                "Score whether OUTPUT is fully supported by CONTEXT. "
                "1.0 = every claim grounded; 0.0 = invents claims. "
                'Return JSON: {"score": <0-1>, "reasoning": "<one sentence>"}.'
            )},
            {"role": "user", "content": f"Context:\n{context}\n\nOutput:\n{output}"},
        ],
    )
    data = json.loads(judgment.choices[0].message.content)
    return data["score"]

## Pytest tests with regression detection

evalcheck's `@eval` decorator wraps a normal pytest test, scores its return value, asserts a hard threshold, and (after the first run is blessed as a baseline) asserts no regression beyond a tolerance.

In [ ]:
%%writefile test_summaries.py
from evalcheck import EvalOutput, eval, faithfulness

from __main__ import summarise

ARTICLE = (
    "Paris, the capital of France, sits on the river Seine. "
    "The Eiffel Tower is its tallest structure."
)

@eval(faithfulness, threshold=0.7)
def test_summary_grounded():
    summary = summarise(ARTICLE)
    return EvalOutput(output=summary, context=ARTICLE)

In [ ]:
!pytest test_summaries.py -v
!evalcheck snapshot --update    # bless the first run
# git add .evalcheck && git commit -m 'chore: bless eval baseline'
# Future PRs fail if faithfulness drops > 0.05 below baseline.

## What's happening under the hood

Every `@eval`-decorated test:

1. Runs the test function as normal pytest.
2. Captures the return value (a string or `EvalOutput`).
3. Calls the metric — for `faithfulness`, that's the OpenAI-as-judge call shown above, just packaged.
4. Compares against a committed baseline. Fails if the drop exceeds `regression_tolerance` (default 0.05).

If you'd rather roll your own without a dependency, the function in step 3 plus a JSON file in your repo gets you 90% of the way there.

## More

- [evalcheck on GitHub](https://github.com/Boiga7/evalcheck)
- [Comparisons vs deepeval, pytest-evals, promptfoo, braintrust](https://github.com/Boiga7/evalcheck/tree/main/docs/comparisons)